In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MultiLabelBinarizer
import ast
from scipy.sparse import csr_matrix

In [2]:
output_path = "../input/"
train_path = output_path + "train/"
val_path = output_path + "val/"

train_df = train_path + "train_fold0_split.csv"
val_df = val_path + "val_fold0_split.csv"
train_emb = train_path + "train_fold0_embeddings.npy"
val_emb = val_path + "val_fold0_embeddings.npy"

y_train_sparse_path = train_path + "Y_train_fold0_sparse.npy"
y_val_sparse_path = val_path + "Y_val_fold0_sparse.npy"

classes_path = output_path + "classes_fold0.npy"
ia_weights_path = "../../general_input/IA.tsv"
final_ia_weights_path = output_path + "ia_weights_fold0.npy"

In [3]:
df_train = pd.read_csv(train_df)
df_val = pd.read_csv(val_df)
X_train = np.load(train_emb)
X_val = np.load(val_emb)

In [4]:
def safe_eval_and_flatten(x):
    if pd.notna(x) and x.strip().startswith('['):
        try:
            # 1. Convert string to list of lists (e.g., "[['GO:1'], ['GO:2', 'GO:3']]")
            evaluated_list_of_lists = ast.literal_eval(x)
            
            # 2. Flatten the list of lists into a single list
            return [term for sublist in evaluated_list_of_lists for term in sublist]
        except:
            # Return empty list if evaluation fails
            return []
    return []

def parse_go_terms_flattened(go_terms_series):
    # Apply the safe evaluation and flattening function to the series
    return go_terms_series.apply(safe_eval_and_flatten)

In [5]:
df_train

,db,id,name,full_name,organism,taxon_x,gen,PE,SV,sequence_x,length,go_terms,aspect,sequence_y,seq_length,taxon_y,rep_id,fold
0,sp,A0A023FBW4,E1142_AMBCJ,Evasin P1142,Amblyomma cajennense,34607,NaN,3,1,MTSHGAVKIAIFAVIALHSIFECLSKPQILQRTDHSTDSDWDPQMC...,97,[['GO:0019958']],['F'],MTSHGAVKIAIFAVIALHSIFECLSKPQILQRTDHSTDSDWDPQMC...,97,34607,A0A023FBW4,4
1,sp,A0A023FBW7,EV546_AMBCJ,Evasin P546,Amblyomma cajennense,34607,NaN,3,1,MKVLLYIAASCLMLLALNVSAENTQQEEEDYDYGTDTCPFPVLANK...,118,[['GO:0019957']],['F'],MKVLLYIAASCLMLLALNVSAENTQQEEEDYDYGTDTCPFPVLANK...,118,34607,A0A023FBW7,1
2,sp,A0A023FDY8,EV974_AMBCJ,Evasin P974,Amblyomma cajennense,34607,NaN,1,1,MKVLLCIAASCLMLLALNVSAENTQQEEQDYDYGTDTCPFPVLANK...,118,[['GO:0019957']],['F'],MKVLLCIAASCLMLLALNVSAENTQQEEQDYDYGTDTCPFPVLANK...,118,34607,A0A023FBW7,1
3,sp,A0A023FF81,E1126_AMBCJ,Evasin P1126,Amblyomma cajennense,34607,NaN,3,1,MTSHSAVRIAIFAVIALHSIFECLSKPQILQRTDKSTDSEWDPQTC...,90,[['GO:0019958']],['F'],MTSHSAVRIAIFAVIALHSIFECLSKPQILQRTDKSTDSEWDPQTC...,90,34607,A0A023FBW4,4
4,sp,A0A023FFB5,EV983_AMBCJ,Evasin P983,Amblyomma cajennense,34607,NaN,3,2,MKASFCVIASCLVVFALKGTAEDTGTEDDFDYGNTGCPFPVLGNYK...,119,[['GO:0019957']],['F'],MKASFCVIASCLVVFALKGTAEDTGTEDDFDYGNTGCPFPVLGNYK...,119,34607,L7M8Z8,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
65749,sp,X2JDY8,RHGF_DROME,Rho GTPase-activating protein Graf,Drosophila melanogaster,7227,Graf,1,1,MGGGKNVRRGLEPLEFEECIVDSPDFRENLNRHEKELDHTSHQIKR...,1025,"[['GO:0140036'], ['GO:0031410'], ['GO:0002092'...","['F', 'C', 'P', 'P', 'C', 'C', 'C', 'F', 'P', ...",MGGGKNVRRGLEPLEFEECIVDSPDFRENLNRHEKELDHTSHQIKR...,1025,7227,X2JDY8,3
65750,sp,X4Y2L4,LHYAL_HIRNI,Hyaluronoglucuronidase,Hirudo nipponia,42736,NaN,1,1,MKEIAVTIDDKNVIASVSESFHGVAFDASLFSPKGLWSFVDITSPK...,489,"[['GO:0033906'], ['GO:0030214']]","['F', 'P']",MKEIAVTIDDKNVIASVSESFHGVAFDASLFSPKGLWSFVDITSPK...,489,42736,Q9FF10,2
65751,sp,X5JA13,SECAA_ARATH,Exocyst complex component SEC10a,Arabidopsis thaliana,3702,SEC10a,1,1,MTEGIRARGPRSSSVNSVPLILDIEDFKGDFSFDALFGNLVNDLLP...,825,"[['GO:0070062'], ['GO:0005829'], ['GO:0005515'...","['C', 'C', 'F', 'C', 'C', 'C', 'C', 'P']",MTEGIRARGPRSSSVNSVPLILDIEDFKGDFSFDALFGNLVNDLLP...,825,3702,X5JB51,1
65752,sp,X5JB51,SECAB_ARATH,Exocyst complex component SEC10b,Arabidopsis thaliana,3702,SEC10b,1,1,MTERIRARGPRSSSVNSVPLILDIEDFKGDFSFDALFGNLVNDLLP...,829,"[['GO:0070062'], ['GO:0005829'], ['GO:0005886'...","['C', 'C', 'C', 'C', 'C', 'C', 'P', 'F']",MTERIRARGPRSSSVNSVPLILDIEDFKGDFSFDALFGNLVNDLLP...,829,3702,X5JB51,1


In [6]:
y_train_lists = parse_go_terms_flattened(df_train['go_terms'])
y_val_lists = parse_go_terms_flattened(df_val['go_terms'])

In [7]:
# Binarizer
mlb = MultiLabelBinarizer(sparse_output=True)

In [8]:
# Codification adjustment
Y_train_sparse = mlb.fit_transform(y_train_lists)

In [9]:
Y_val_sparse = mlb.transform(y_val_lists)

/home/maria/miniconda3/envs/torch_env_kaggle_wi/lib/python3.11/site-packages/sklearn/preprocessing/_label.py:909: UserWarning: unknown class(es) ['GO:0000041', 'GO:0000252', 'GO:0000374', 'GO:0000401', 'GO:0001095', 'GO:0001172', 'GO:0001173', 'GO:0001512', 'GO:0001576', 'GO:0001607', 'GO:0001631', 'GO:0001632', 'GO:0001677', 'GO:0001695', 'GO:0001705', 'GO:0001770', 'GO:0001790', 'GO:0001792', 'GO:0001806', 'GO:0001807', 'GO:0001808', 'GO:0001815', 'GO:0001830', 'GO:0001868', 'GO:0001910', 'GO:0001980', 'GO:0001985', 'GO:0001990', 'GO:0001993', 'GO:0002014', 'GO:0002041', 'GO:0002112', 'GO:0002132', 'GO:0002152', 'GO:0002172', 'GO:0002178', 'GO:0002194', 'GO:0002206', 'GO:0002268', 'GO:0002364', 'GO:0002412', 'GO:0002414', 'GO:0002420', 'GO:0002443', 'GO:0002451', 'GO:0002460', 'GO:0002462', 'GO:0002476', 'GO:0002477', 'GO:0002538', 'GO:0002561', 'GO:0002579', 'GO:0002617', 'GO:0002643', 'GO:0002676', 'GO:0002699', 'GO:0002715', 'GO:0002828', 'GO:0002834', 'GO:0002840', 'GO:0002878', 

In [10]:
print("--- Codification results ---")
print(f"Label shape (Y_train): {Y_train_sparse.shape}")
print(f"Object type: {type(Y_train_sparse)}") 
# <class 'scipy.sparse.csr.csr_matrix'>
print(f"RAM used (aprox): {Y_train_sparse.data.nbytes / 1024**2:.2f} MB")

--- Codification results ---
Label shape (Y_train): (65754, 24373)
Object type: <class 'scipy.sparse._csr.csr_matrix'>
RAM used (aprox): 3.27 MB


In [11]:
# Save Y_train_sparse and Y_val_sparse
np.save(y_train_sparse_path, Y_train_sparse)
np.save(y_val_sparse_path, Y_val_sparse)

# Save classes.npy (used later)

In [12]:
from tqdm import tqdm

In [13]:
# We use a SET to automatically store only unique terms
unique_classes = set()

# Iterate row by row to save RAM
for row in tqdm(df_train['go_terms']):
    terms = safe_eval_and_flatten(row)
    unique_classes.update(terms)

100%|██████████| 65754/65754 [00:02<00:00, 26473.53it/s]


In [14]:
sorted_classes = sorted(list(unique_classes))
classes_array = np.array(sorted_classes)

print(f"Number of unique GO terms in training set: {len(sorted_classes)}")

Number of unique GO terms in training set: 24373


In [15]:
num_cols_matrix = Y_train_sparse.shape[1]
    
if len(classes_array) == num_cols_matrix:
    print("OK! The number of classes matches the trained matrix.")
    
    # Save so we don't need to repeat this ever again
    np.save(classes_path, classes_array)
    print(f"File saved to: {output_path}classes.npy")
else:
    print(f"WARNING: Size mismatch.")
    print(f"Unique terms found in CSV: {len(classes_array)}")
    print(f"Columns in sparse matrix: {num_cols_matrix}")
    print("Check if data filtering was identical.")

OK! The number of classes matches the trained matrix.
File saved to: ../input/classes.npy


# Save IA_weights (used on validation)

In [16]:
df_weights = pd.read_csv(ia_weights_path, sep="\t", names=["go_terms", "ia"])
df_weights.head()

,go_terms,ia
0,GO:0000001,0.000000
1,GO:0000002,2.849666
2,GO:0000011,0.137504
3,GO:0000012,6.038630
4,GO:0000017,0.514573


In [17]:
ia_score_map = df_weights.set_index('go_terms')['ia'].to_dict()

In [18]:
MIN_IA_SCORE = 0.0

In [19]:
LABEL_COUNT = len(mlb.classes_)
LABEL_COUNT

24373

In [20]:
ia_weights_ordered = np.zeros(LABEL_COUNT, dtype=np.float32)

In [21]:
for i, go_term in enumerate(mlb.classes_):
    weight = ia_score_map.get(go_term, MIN_IA_SCORE)
    ia_weights_ordered[i] = weight

In [22]:
ia_weights_ordered.shape

(24373,)

In [23]:
np.save(final_ia_weights_path, ia_weights_ordered)